In [0]:
from pyspark.sql.functions import sum,col,greatest,avg,coalesce,lit,trim,upper,when,lit,min,max,count,dense_rank,row_number,rank
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
from pyspark.sql.window import Window

In [0]:
# 1. Create a Spark DataFrame with the given employee schema and load the 1000 records. --Loaded with some Nulls as well in file

schema=StructType([
StructField("employee_id",IntegerType(),True),
StructField("employee_name",StringType(),True),
StructField("department",StringType(),True),
StructField("state",StringType(),True),
StructField("salary",IntegerType(),True),
StructField("age",IntegerType(),True),
StructField("bonus",IntegerType(),True),
])
df= spark.read.format("csv").options(header=True,inferSchema=True,delimiter=",",nullValue="None").schema(schema).load("/Volumes/workspace/sanjay_volume/june2026_rowdata/OfficeDataProject_withNulls.csv")
df.cache()    #Stores DataFrame in memory.
df.show()

In [0]:
# 2. Display the first 10 records from the employee DataFrame.

df_10=df.show(10)

In [0]:
# 3. Print the schema of the DataFrame.
df.printSchema()

In [0]:
# 4. Count the total number of records in the DataFrame.
df.count()

In [0]:
# 5. Check the number of distinct departments available.
df.select("department").distinct().count()


In [0]:
# 6. Find all distinct states from the employee DataFrame.
df.select("state").distinct().show()

In [0]:
#7. Check and handle null values in all columns.
from pyspark.sql.functions import sum,col,greatest,avg,coalesce,lit
#Nulls counts per column
nullcounts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
nullcounts.show()

# containing atleast one null value
rows_with_nulls = df.filter(
    greatest(*[col(c).isNull().cast("int") for c in df.columns]) == 1
)
rows_with_nulls.show()

# handling nulls
# df.agg(avg("salary")).collect()      
# df.agg(avg("salary")).collect()[0]   
# df.agg(avg("salary")).collect()[0][0]

avg_salary = df.agg(avg("salary")).collect()[0][0]
avg_age=df.agg(avg("age")).collect()[0][0]
avg_bonus=df.agg(avg("bonus")).collect()[0][0]
cleaned_df = (df.fillna({"employee_name": "Unknown"})
              .fillna({"department":"Unassigned"})
              .fillna({"state": "unknown"})
              .fillna({"salary":int(avg_salary)})
              .fillna({"age":int(avg_age)})
              .fillna({"bonus":int(avg_bonus)})
              )
cleaned_df.show()





In [0]:
###8. Replace null salary values with the average salary.
#method-1
avg_salary = df.agg(avg("salary")).collect()[0][0]
df_sal = df.fillna({"salary": int(avg_salary)})
df_sal.display()

#method-2
df_sal1=df.withColumn("salary",coalesce(col("salary"),lit(int(avg_salary))))
df_sal1.display()


In [0]:
cleaned_df.show()

In [0]:
# 9. Remove records where employee_name is null.
#method-1
df_emp=df.filter(df.employee_name.isNotNull())
#method-2
df_emp=df.na.drop(subset=["employee_name"])
df_emp.display()

In [0]:
# 10. Remove duplicate employee records based on employee_id.
df_dedup=df.dropDuplicates(["employee_id"])
df_dedup.display()

In [0]:
# 11. Rename the column employee_name to emp_name.
df_renamed=df.withColumnRenamed("employee_name","emp_name")
df_renamed.show()

In [0]:
# 12. Change salary data type to integer or double.
df_cast=df.withColumn("salary",col("salary").cast("double"))
df_cast.display()

In [0]:
# 13. Trim spaces from employee_name values.
df_trim=df.withColumn("employee_name",trim(col("employee_name")))
df_trim.show()



In [0]:
# 14. Convert employee_name values into uppercase.
df_uppr=df.withColumn("employee_name",upper(col("employee_name")))
df_uppr.show()

In [0]:
# 15. Create a new column annual_salary by calculating salary * 12.
df_annual=df.withColumn("annual_salary",col("salary")*12)
df_annual.show()

In [0]:
# 16. Create a new column total_compensation by adding salary and bonus.
df_compn=df.withColumn("total_compensation",col("salary")+col("bonus"))
df_compn.show()

In [0]:
# 17. Increase salary by 10 percent for all employees and create a new column updated_salary.
df_uptsal=df.withColumn("updated_salary",col("salary")*1.10)
df_uptsal.show()

In [0]:
# 18. Create an age_category column: Young (<30), Mid (30-50), Senior (>50).
df_agecat= df.withColumn("age_category", 
                when(col("age")<30,"Young")
                .when((col("age") >= 30) & (col("age") <= 50),"Mid")
				.otherwise("Senior"))
df_agecat.show()

In [0]:
#19. Create a salary_band column based on salary ranges.
df_salband= df.withColumn("salary_band", 
                when(col("salary")<50000,"Low")
                .when((col("salary") >= 50000) & (col("salary") <= 100000),"Medium")
                .when(col("salary") > 100000,"High"))
df_salband.show()

In [0]:
# 20. Filter employees whose salary is greater than 50000.
df_high=df.filter(df.salary>5000)
df_high.show()

In [0]:
# 21. Find employees who belong to a specific department.
df_spDp=df.filter(df.department=="HR")
df_spDp.show()



In [0]:
# 22. Filter employees from multiple states using conditions.
df_states = df.filter(col("state").isin("CA", "AK", "LA"))
df_states.show()


In [0]:
# 23. Sort employees based on salary descending order.
df=df.orderBy(col("salary").desc())
df.show()


In [0]:
# 24. Sort employees by department and salary.
df=df.orderBy(col("department"),col("salary"))
df.show()

In [0]:
# 25. Find the top 10 highest paid employees.
df_top10=df.orderBy(col("salary").desc())
df_top10.show(10)

In [0]:
# 26. Find the bottom 10 lowest paid employees.
df_bottom10=df.orderBy(col("salary").asc())
df_bottom10.show(10)


In [0]:
# 27. Calculate average salary of all employees.

avg_salary=df.agg(avg("salary")).collect()[0][0]
df_avg=df.withColumn("average_salary",lit(avg_salary))
df_avg.show()

print("=" * 60)

df_avg_salary = df.select(avg("salary").alias("average_salary"))
df_avg_salary.show()


In [0]:
# 28. Calculate minimum and maximum salary.
df_min_mx=df.select(min("salary").alias("min_salary"),max("salary").alias("max_salary"))
df_min_mx.show()

In [0]:
# 29. Find total bonus paid by the company.
df_totalBonus=df.select(sum("bonus").alias("Total_bonus"))
df_totalBonus.show()

In [0]:
# 30. Group employees by department and calculate employee count.
df_grp = df.groupBy("department").agg(count("*").alias("CountOfEmployee"))
df_grp.show()

In [0]:
# 31. Find average salary department-wise.

df_avg=df.groupBy("department").agg(avg("salary").alias("avg_sal"))
df_avg.show()

In [0]:
# 32. Find maximum salary in each department.
df_maxSal=df.groupBy("department").agg(max("salary").alias("max_salary"))
df_maxSal.show()

In [0]:
# 33. Find state-wise employee count.
df_stCount=df.groupBy("state").agg(count("*").alias("CountOfEmployee"))
df_stCount.show()

In [0]:
df_totCp=df.groupBy("department").agg(sum("salary").alias("total_compensation"))
df_totCp.show()

In [0]:
# 35. Use groupBy and aggregation functions to generate department salary statistics.
df_salStatic=df.groupBy("department").agg(sum("salary").alias("Total_sal"),min("salary").alias("min_salary"),max("salary").alias("max_salary"),avg("salary").alias("AVG_Sal"))
df_salStatic.show()

In [0]:
# 36. Create a temporary view from the DataFrame and query it using Spark SQL.
df.createOrReplaceTempView("officeDataTemp")


In [0]:
%sql
select * from  officeDataTemp

In [0]:
%sql
--37. Write a Spark SQL query to find employees older than 40 years.

select employee_id,age from officeDataTemp where age >40

In [0]:
# 38. Use window functions to find employee salary rank within each department.
win_spec=Window.partitionBy("department").orderBy(col("salary").desc())

df_rn=df.withColumn("rnk",dense_rank().over(win_spec))
df_rn.show()

In [0]:
# 39. Find the highest salary employee in each department using window function.
df_higsal=df.withColumn("salRank",dense_rank().over(win_spec)).filter(col("salRank")==1)
df_higsal.show()

In [0]:
# 40. Create a final cleaned DataFrame and write it as a Delta table
# using cleaned_df where I handle null values
cleaned_df.write.format("delta").mode("append").save("/Volumes/workspace/sanjay_volume/silver_layer/Cleanofficedata")

In [0]:
dbutils.fs.ls("/Volumes/workspace/sanjay_volume/silver_layer/Cleanofficedata")




In [0]:

# 2. Implement logging while performing transformations.
import logging

# This sets up a logger that prints messages to the notebook output
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("office_pipeline")

# Test it
logger.info("Logger is ready")
logger.warning("This is a warning test")
logger.error("This is an error test")

In [0]:
# -- Create the Log Writer Function
# -- This function writes one log entry as a row into a Delta table.
from datetime import datetime
from pyspark.sql import Row

# Where your log table will be saved
LOG_PATH = "/Volumes/workspace/sanjay_volume/silver_layer/audit_logs/"

def write_log(check_name, status, message, record_count=None):
    """
    check_name  : name of the check  e.g. 'null_check'
    status      : INFO / WARNING / ERROR
    message     : what happened
    record_count: how many rows were affected (optional)
    """
    # Print to notebook console via logger
    logger.info(f"[{check_name}] {status} — {message}")

    # Build one row
    log_row = Row(
        check_name   = check_name,
        status       = status,
        message      = message,
        record_count = record_count,
        logged_at    = datetime.now().isoformat()
    )

    # Write that row to Delta table
    spark.createDataFrame([log_row]) \
         .write.format("delta") \
         .mode("append") \
         .save(LOG_PATH)

# Test it
write_log("test_check", "INFO", "Log writer is working", record_count=0)
print("Log written successfully")

In [0]:
%sql
select * from delta.`/Volumes/workspace/sanjay_volume/silver_layer/audit_logs/`

In [0]:
# Write the 4 Checks
from pyspark.sql import functions as F

# ── CHECK 1 : Row Count ──────────────────────────────────────
# Makes sure the DataFrame is not empty
def check_row_count(df, min_expected=1):
    count = df.count()
    if count < min_expected:
        write_log("row_count_check", "ERROR",
                  f"Only {count} rows found. Expected at least {min_expected}.", count)
    else:
        write_log("row_count_check", "INFO",
                  f"Row count is {count}. Looks good.", count)


# ── CHECK 2 : Null Check ─────────────────────────────────────
# Checks specific columns for NULL values
def check_nulls(df, columns_to_check):
    for col_name in columns_to_check:
        null_count = df.filter(F.col(col_name).isNull()).count()
        if null_count > 0:
            write_log("null_check", "WARNING",
                      f"Column '{col_name}' has {null_count} NULL value(s).", null_count)
        else:
            write_log("null_check", "INFO",
                      f"Column '{col_name}' — no NULLs found.", 0)


# ── CHECK 3 : Duplicate Check ────────────────────────────────
# Checks if employee_id has any duplicate values
def check_duplicates(df, pk_col="employee_id"):
    total    = df.count()
    distinct = df.select(pk_col).distinct().count()
    dupes    = total - distinct
    if dupes > 0:
        write_log("duplicate_check", "ERROR",
                  f"{dupes} duplicate(s) found in column '{pk_col}'.", dupes)
    else:
        write_log("duplicate_check", "INFO",
                  f"No duplicates found in '{pk_col}'.", 0)


# ── CHECK 4 : Range Check ────────────────────────────────────
# Checks if salary and age values fall within a valid range
def check_range(df, col_name, min_val, max_val):
    out_of_range = df.filter(
        F.col(col_name).isNotNull() &
        ((F.col(col_name) < min_val) | (F.col(col_name) > max_val))
    ).count()
    if out_of_range > 0:
        write_log("range_check", "WARNING",
                  f"Column '{col_name}': {out_of_range} value(s) outside [{min_val}–{max_val}].",
                  out_of_range)
    else:
        write_log("range_check", "INFO",
                  f"Column '{col_name}': all values within [{min_val}–{max_val}].", 0)

In [0]:
# Run All Checks
print("Starting checks...\n")

check_row_count(df, min_expected=1)

check_nulls(df, columns_to_check=["employee_id", "employee_name","department","state", "salary", "age","bonus"])

check_duplicates(df, pk_col="employee_id")

check_range(df, col_name="salary", min_val=1000,  max_val=200000)
check_range(df, col_name="age",    min_val=18,    max_val=65)

print("\nAll checks done.")

In [0]:
# --  View the Log Table
log_df = spark.read.format("delta").load(LOG_PATH)
log_df.orderBy("logged_at", ascending=False).show(truncate=False)



In [0]:
# To see only warnings and errors:
log_df.filter(F.col("status").isin("WARNING", "ERROR")).show(truncate=False)

In [0]:
# 4. Handle schema evolution scenario for incoming employee data.
cleaned_df.write.format("delta").mode("append").option("mergeSchema", "true").save("/Volumes/workspace/sanjay_volume/ofcdata/june24/officeClean1")

In [0]:
# 5. Create a daily incremental processing logic using employee_id.
# 1. Read incoming
df_incoming = spark.read.format("csv").options(header=True, nullValue="None").schema(schema).load("/Volumes/workspace/sanjay_volume/june2026_rowdata/new/OfficeDataProject_withNulls.csv")
df_incoming.show()
# 2. Read existing delta which I already placed in above cells while wrote my final df in delta
df_existing = spark.read.format("delta").load("/Volumes/workspace/sanjay_volume/silver_layer/Cleanofficedata")
df_existing.show()

In [0]:
# 3. Find new records only
df_new = df_incoming.join(df_existing,on="employee_id",how="left_anti")
df_new.show()
# 4. Append to delta table
df_new.write.format("delta").mode("append").save("/Volumes/workspace/sanjay_volume/silver_layer/Cleanofficedata")


In [0]:
%sql
select * from delta.`/Volumes/workspace/sanjay_volume/silver_layer/Cleanofficedata`
